# ROI classification: label, train, apply

1. Demix: `raw -> demixing_results.hdf5` (one file per session, never modified afterwards).
2. Label ROIs in the GUI, or apply a classifier you already have. Labels live in a sidecar
   `demixing_results.labels.hdf5` next to the results.
3. Train: the classifier is saved next to the hdf5 and its path is recorded in the sidecar.
4. Use the labels / predictions downstream.

Every button in the GUI is also a method on `ClassificationVis`; `vis.wait()` blocks until
background work (loading, train, classify) has finished.

In [1]:
import numpy as np
import h5py
import torch
from masknmf.visualization import ClassificationVis
from masknmf.classification import RoicatClassifier
from masknmf.demixing.demixing_results import DemixingResults

import masknmf
import torch
import sys
import numpy as np

from typing import *
import fastplotlib as fpl
import masknmf
import scipy
import h5py
import scipy.sparse
import numpy as np
import torch
import os
import roicat
from masknmf.multisession import RoicatDataAdapter
from masknmf.classification import RoicatClassifier
from functools import partial

import tempfile

rendercanvas could not load some backends:
terminal: No module named 'blessed'
Rendercanvas selected anywidget backend because running on Jupyter.
To silence this warning, use a fully namespaced name.


In [3]:
file_list = ['file_1.hdf5', 'file_2.hdf5']

## 1. Label

Keys `1-9` label the current ROI, `0` clears, up/down move, `u` jumps to the next unlabeled ROI, `h` opens help.
Labels autosave to `<results>.labels.hdf5`: `class_labels`, `label_names`, `roi_masks`.

In [4]:
vis = ClassificationVis.from_masknmf(file_list, label_names=["soma", "dendrite", "junk"])
vis.show()

/data/home/app2139/masknmf-toolbox/masknmf/utils/_serialization.py:191: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  ans[key] = torch.sparse_coo_tensor(


### Plot a representative example of each label from each of these categories 

In [ ]:
import matplotlib.pyplot as plt

labels, names = vis.class_labels, vis.label_names
fig, axes = plt.subplots(1, len(names), figsize=(3 * len(names), 3))
for ax, (i, name) in zip(np.atleast_1d(axes), enumerate(names)):
    sel = labels == i
    ax.imshow(vis.roi_images[sel].mean(axis=0) if sel.any() else np.zeros(vis.roi_images.shape[1:]), cmap="gray")
    ax.set_title(f"{name} (n={sel.sum()})")
    ax.axis("off")

## 2. Train

Needs every ROI labeled, at least 2 per class. Same as the train button: saves to `vis.classifier_path`
(default `classifier.roicat_classifier` next to the first file) and records it in each sidecar
(`classifier_path`, `classifier_history`).

In [ ]:
vis.train()  # or vis.train("path/to/my_classifier")
vis.wait()
classifier_path = vis.classifier_path
print(vis.status)

Without the GUI (labels come from the files, or set `clf.labels` yourself):

In [ ]:
clf = RoicatClassifier.from_masknmf(file_list)  # picks up the labels stored in the files
clf.train(num_workers=0)
classifier_path = clf.save(classifier_path)      # also writes classifier.training.json: labels + training files

## 3. Apply to another session

Unlabeled ROIs take the prediction, existing labels are kept; the `pred` column shows name + confidence
(red where it disagrees with your label). Predictions are saved as `class_predictions`,
`class_probabilities` and `classified_with`.

In [ ]:
new_files = ['some_file.hdf5']
vis2 = ClassificationVis.from_masknmf(new_files)
vis2.select_classifier(classifier_path)  
vis2.classify()                        
vis2.show()

Without the GUI:

In [ ]:
clf = RoicatClassifier.from_disk(classifier_path)
ids, pred_names, probs = clf.classify(new_files, write=True)  # one entry per session; write=True stores them in the files
pred_names[0][:10], probs[0].max(axis=1)[:10]

## 4. Downstream

In [ ]:
from masknmf.demixing.labels import labels_path, read_labels

labels, names = read_labels(new_files[0])  # (num_rois,) int64 with -1 = unlabeled, class names
with h5py.File(labels_path(new_files[0]), "r") as f:  # demixing_results.labels.hdf5
    masks = f["roi_masks"][()]            # (num_rois, 36, 36) float32
    preds = f["class_predictions"][()]    # (num_rois,) index into names
    conf = f["class_probabilities"][()]   # (num_rois,) confidence of preds
    print(f["classifier_path"][()].decode(), [h.decode() for h in f["classifier_history"][()]])

In [ ]:
dmr = DemixingResults.from_hdf5(new_files[0])  # untouched by labeling
keep = np.flatnonzero(labels == names.index("soma"))
traces = dmr.c[:, keep]                                      # (num_frames, num_soma)
footprints = dmr.a.index_select(1, torch.as_tensor(keep))    # (pixels, num_soma), sparse
traces.shape, footprints.shape

## Command line

`classification a.hdf5 [b.hdf5 ...] --labels soma,dendrite,junk [--classifier path]`

A `.npy` stack of `(num_rois, Y, X)` masks works too; its labels go to `<path>.labels.npz`.